In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
TDMPC2_DIR = WORKDIR / "tdmpc2"
DATA_DIR = WORKDIR / "data" / "dmc_expert"

WORKDIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not R2DREAMER_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "mamba3-rssm-experiment",
            "https://github.com/hanswalker/r2dreamer.git",
            str(R2DREAMER_DIR),
        ],
        check=True,
    )

if not TDMPC2_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/nicklashansen/tdmpc2.git",
            str(TDMPC2_DIR),
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "numpy==1.26.0",
        "pyyaml",
        "zarr",
        "huggingface_hub",
        "dm_control==1.0.28",
        "mujoco==3.3.0",
        "omegaconf",
        "hydra-core",
        "tensordict",
        "torchrl",
        "kornia",
        "termcolor",
        "tqdm",
        "pandas",
        "moviepy",
        "imageio",
        "imageio-ffmpeg",
        "h5py",
    ],
    check=True,
)

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

os.chdir(R2DREAMER_DIR)
if str(R2DREAMER_DIR) not in sys.path:
    sys.path.insert(0, str(R2DREAMER_DIR))

collector_candidates = [
    R2DREAMER_DIR / "collect_dmc_expert_data.py",
    R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py",
]
COLLECTOR = next((path for path in collector_candidates if path.exists()), None)
if COLLECTOR is None:
    raise FileNotFoundError(
        "Missing collect_dmc_expert_data.py. Expected it at either "
        f"{collector_candidates[0]} or {collector_candidates[1]}."
    )

import torch

print("R2Dreamer:", R2DREAMER_DIR)
print("TD-MPC2:", TDMPC2_DIR)
print("Data:", DATA_DIR)
print("Collector:", COLLECTOR)
print("CUDA:", torch.cuda.is_available())


In [ ]:
CONFIG_PATH = DATA_DIR / "collect_dmc_expert_smoke.yaml"
CONFIG_PATH.write_text(
    f"""
tdmpc2_root: {TDMPC2_DIR}
output_dir: {DATA_DIR}

tasks:
  - cartpole/swingup

num_episodes: 2
checkpoint_seed: 1
seed: 1

action_repeat: 2
max_episode_steps: 500

save_images: false
image_size: 64

resume: true
""".strip()
    + "\n",
    encoding="utf-8",
)

cmd = [sys.executable, str(COLLECTOR), "--config", str(CONFIG_PATH)]
subprocess.run(cmd, check=True)
